In [ ]:
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier


# =============================================================================
# 1. CONFIGURAÇÕES E CONSTANTES GLOBAIS
# =============================================================================
CAMINHO = '/content/SPDadosCriminais_GERAL.csv'

# Mesma semente do mlp.py e do reg_log.py para garantir divisões idênticas dos dados
SEED = 1000


# =============================================================================
# 2. CARREGAMENTO E LIMPEZA INICIAL DOS DADOS
# =============================================================================
df = pd.read_csv(CAMINHO, sep=',', encoding='utf-8-sig', low_memory=False)
print(f"-> Dimensões originais do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")

# Conversão numérica segura das coordenadas
df['LAT_NUM'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
df['LON_NUM'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')

# Remoção de coordenadas nulas ou (0, 0) inválidas
df_limpo = df[
    df['LAT_NUM'].notna() & df['LON_NUM'].notna() &
    (df['LAT_NUM'] != 0) & (df['LON_NUM'] != 0)
].copy()

# Tratamento temporal: hora e dia da semana
df_limpo['HORA_LIMPA'] = df_limpo['HORA_OCORRENCIA_BO'].astype(str).str.strip().str.split(':').str[0]
df_limpo['HORA_INT'] = pd.to_numeric(df_limpo['HORA_LIMPA'], errors='coerce').fillna(12)
df_limpo['DATA_DT'] = pd.to_datetime(df_limpo['DATA_OCORRENCIA_BO'], errors='coerce')
df_limpo['DIA_SEMANA'] = df_limpo['DATA_DT'].dt.dayofweek.fillna(0)

print(f"-> Dimensões após limpeza de coordenadas: {df_limpo.shape[0]} linhas")

# Amostragem para viabilidade computacional rápida (800.000 amostras)
if len(df_limpo) > 800000:
    df_sample = df_limpo.sample(n=800000, random_state=SEED).copy()
else:
    df_sample = df_limpo.copy()


# =============================================================================
# 3. DIVISÃO ESTRITA DE TREINO/TESTE (EVITA DATA LEAKAGE)
# =============================================================================
df_train, df_test = train_test_split(df_sample, test_size=0.30, random_state=SEED)

# Arredondamento das coordenadas para criar blocos/zonas de vizinhança
for dataset in [df_train, df_test]:
    dataset['LAT_ROUND'] = dataset['LAT_NUM'].round(2)
    dataset['LON_ROUND'] = dataset['LON_NUM'].round(2)


# =============================================================================
# 4. ENGENHARIA DO ALVO (BASEADA APENAS NO TREINO)
# =============================================================================
# Frequência de crimes por zona calculada somente no conjunto de treino
contagem_zona_treino = (
    df_train.groupby(['LAT_ROUND', 'LON_ROUND'])
    .size()
    .reset_index(name='FREQ_CRIMES')
)

# Limiar de perigo: mediana das frequências no treino
limiar_perigo = contagem_zona_treino['FREQ_CRIMES'].median()

# Mapeando de volta ao treino
df_train = pd.merge(df_train, contagem_zona_treino, on=['LAT_ROUND', 'LON_ROUND'], how='left')
df_train['LOCAL_PERIGOSO'] = (df_train['FREQ_CRIMES'] > limiar_perigo).astype(int)

# Mapeando ao teste de forma "cega" (somente com o que foi aprendido no treino)
df_test = pd.merge(df_test, contagem_zona_treino, on=['LAT_ROUND', 'LON_ROUND'], how='left')
df_test['FREQ_CRIMES'] = df_test['FREQ_CRIMES'].fillna(0)
df_test['LOCAL_PERIGOSO'] = (df_test['FREQ_CRIMES'] > limiar_perigo).astype(int)

print("\nDistribuição de Classes no Treino:", np.bincount(df_train['LOCAL_PERIGOSO']))
print("Distribuição de Classes no Teste: ", np.bincount(df_test['LOCAL_PERIGOSO']))


# =============================================================================
# 5. ENGENHARIA DE RECURSOS ESPACIAIS (PROJEÇÃO 3D)
# =============================================================================
for dataset in [df_train, df_test]:
    lat_rad = np.radians(dataset['LAT_NUM'])
    lon_rad = np.radians(dataset['LON_NUM'])
    dataset['COORD_X'] = np.cos(lat_rad) * np.cos(lon_rad)
    dataset['COORD_Y'] = np.cos(lat_rad) * np.sin(lon_rad)
    dataset['COORD_Z'] = np.sin(lat_rad)

# Mesmas features finais utilizadas no pipeline consolidado do mlp.py
features_finais = ['COORD_X', 'COORD_Y', 'COORD_Z', 'HORA_INT', 'DIA_SEMANA']

X_train = df_train[features_finais]
y_train = df_train['LOCAL_PERIGOSO']
X_test = df_test[features_finais]
y_test = df_test['LOCAL_PERIGOSO']

# Obs.: o XGBoost é baseado em árvores de decisão, portanto NÃO requer
# escalonamento das features (diferente do MLP e da Regressão Logística).
# Usamos exatamente as mesmas features brutas para manter a comparação justa.


# =============================================================================
# 6. BUSCA DE HIPERPARÂMETROS (GridSearchCV)
# =============================================================================
# Razão de desbalanceamento (nº de negativos / nº de positivos) calculada no treino.
# É usada no parâmetro scale_pos_weight, que compensa a sub-representação da
# classe minoritária (alto risco) durante as rodadas de boosting.
ratio_desbalanceamento = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"\n-> Razão de desbalanceamento (neg/pos) no treino: {ratio_desbalanceamento:.2f}")

# Espaço de busca para o XGBoost:
#   - n_estimators     : número de árvores (rodadas de boosting)
#   - max_depth        : profundidade máxima de cada árvore (controla a complexidade)
#   - learning_rate    : taxa de aprendizado (menor = modelos mais robustos)
#   - min_child_weight : soma mínima do hessian nos nós filhos (regularização)
#   - subsample        : fração de amostras usada por árvore (variedade dos modelos)
#   - colsample_bytree : fração de features usada por árvore (variedade dos modelos)
#   - scale_pos_weight : peso da classe positiva (1 = ignora desbalanceamento)
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'min_child_weight': [1, 3],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [1.0, ratio_desbalanceamento],
}

total_combinacoes = int(np.prod([len(v) for v in param_grid.values()]))
print(f"-> Total de {total_combinacoes} combinações de hiperparâmetros serão testadas (3-fold CV).")

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',      # algoritmo otimizado para CPU (ideal para o Google Colab)
    random_state=SEED,
    n_jobs=1,                # evita paralelismo aninhado com o GridSearchCV
    verbosity=0,
)

print("Iniciando a busca de hiperparâmetros com GridSearchCV (3-fold CV)...")
inicio_busca = time.time()
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train, y_train)
tempo_busca = time.time() - inicio_busca

print("\nMelhores parâmetros encontrados:", grid_search.best_params_)
print(f"Melhor acurácia de validação cruzada: {grid_search.best_score_:.4f}")
print(f"Tempo total da busca de hiperparâmetros: {tempo_busca / 60:.1f} minutos")


# =============================================================================
# 7. AVALIAÇÃO DO MODELO OTIMIZADO NO CONJUNTO DE TESTE
# =============================================================================
best_xgb = grid_search.best_estimator_

y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("DESEMPENHO DO XGBOOST OTIMIZADO")
print("=" * 60)
print(f"Acurácia Final no Teste: {accuracy_score(y_test, y_pred):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred))


# =============================================================================
# 8. VISUALIZAÇÕES: MATRIZ DE CONFUSÃO E CURVA ROC
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Baixo Risco (0)', 'Alto Risco (1)'],
            yticklabels=['Baixo Risco (0)', 'Alto Risco (1)'])
axes[0].set_title('Matriz de Confusão - XGBoost')
axes[0].set_ylabel('Classe Real')
axes[0].set_xlabel('Classe Predita')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='green', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aleatório')
axes[1].set_title('Curva ROC - XGBoost')
axes[1].set_xlabel('Taxa de Falsos Positivos')
axes[1].set_ylabel('Taxa de Verdadeiros Positivos')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


# =============================================================================
# 9. IMPORTÂNCIA DAS FEATURES (GANHO DO XGBOOST E PERMUTAÇÃO)
# =============================================================================
# O XGBoost expõa a importância baseada no ganho médio (gain) acumulado nos
# nós divididos por cada feature ao longo de todas as árvores treinadas.
gain_df = pd.DataFrame({
    'Feature': features_finais,
    'Ganho_Medio': best_xgb.feature_importances_,
}).sort_values(by='Ganho_Medio', ascending=False)

print("\nImportância das features (ganho médio do XGBoost):")
print(gain_df.to_string(index=False))

# Permutation Importance (para permitir comparação direta com o mlp.py e o reg_log.py)
print("\nCalculando a importância das features por permutação no conjunto de teste...")
result = permutation_importance(
    best_xgb, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1
)

importances_df = pd.DataFrame({
    'Feature': features_finais,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std,
}).sort_values(by='Importance_Mean', ascending=False)

print("\nImportância por permutação:")
print(importances_df.to_string(index=False))

# Plotando os dois tipos de importância lado a lado
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(x='Ganho_Medio', y='Feature', data=gain_df, hue='Feature',
            palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Importância por Ganho Médio - XGBoost')
axes[0].set_xlabel('Ganho Médio Acumulado')
axes[0].set_ylabel('Variável')
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.barplot(x='Importance_Mean', y='Feature', data=importances_df, hue='Feature',
            palette='magma', legend=False, ax=axes[1])
axes[1].set_title('Importância das Variáveis (Permutation Importance)')
axes[1].set_xlabel('Redução Média na Acurácia (Impacto)')
axes[1].set_ylabel('Variável')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


# =============================================================================
# 10. RESUMO FINAL PARA COMPARAÇÃO COM O MLP E A REGRESSÃO LOGÍSTICA
# =============================================================================
print("\n" + "=" * 60)
print("RESUMO FINAL - XGBOOST")
print("=" * 60)
print(f"Melhores hiperparâmetros : {grid_search.best_params_}")
print(f"Acurácia CV (treino)     : {grid_search.best_score_:.4f}")
print(f"Acurácia no teste        : {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC no teste             : {roc_auc:.4f}")
print("=" * 60)
